# Consistent Hashing — The Ring Behind Sharding & Caches

## 🧠 Mental Model

> **Imagine a clock face. Both servers and keys sit on this circle.
> A key "belongs to" the first server clockwise from it.
> Adding/removing a server only affects one arc — ~1/N keys move, not all.**

### ❌ WHY naive `hash(key) % N` fails at scale

```
Servers = ["s1", "s2", "s3"]           Servers = ["s1", "s2", "s3", "s4"]
key "user:42" → hash % 3 = 1 → s2     key "user:42" → hash % 4 = 3 → s4  ← DIFFERENT!
```

Adding ONE server remaps ~75% of keys (N-1)/N → every cache simultaneously misses
→ DB gets hit by ALL traffic → cascading failure.

### ✅ HOW Consistent Hashing fixes it

```
1. Hash each server name to a position on a ring [0, 2^32)
2. Hash each key to a ring position
3. Key belongs to the first SERVER clockwise from its position
4. Remove s3: only the keys between s2 and s3 move to s4 (~1/N total)
```

### Virtual Nodes (v-nodes)

Without v-nodes, 3 servers each own ~1/3 of the ring — in theory. In practice,
`hash("server3")` might land at position 95%, leaving server1 with 95% of traffic.
**V-nodes = each server gets K random ring positions**, smoothing the distribution.
Cassandra uses 256 v-nodes per server.

### 🌍 Where Consistent Hashing is Used in Production

| System | Usage |
|---|---|
| Amazon DynamoDB | Partition assignment across nodes |
| Apache Cassandra | Virtual nodes — 256 ring positions per server |
| Redis Cluster | 16,384 hash slots distributed across nodes |
| Nginx/HAProxy | `hash $request_uri consistent` upstream directive |
| Memcached | libketama consistent hashing library |

### ⚠️ Gotchas

- **Hot keys still exist** — consistent hashing distributes *ownership*, not *traffic*.
  A viral product page always hits the same cache node. Shard that key explicitly.
- **More v-nodes = more ring map memory** in the coordinator.
- **Client-side vs proxy**: Cassandra driver handles hashing client-side (no coordinator RTT).
  DynamoDB uses a coordinator service. Different failure modes.

---
## ❌ Before — Naive Modulo Hashing

---
## ❌ Before — Naive `hash(key) % N`: everything remaps when N changes

**Without consistent hashing:**
```python
def assign(key, servers):
    return servers[hash(key) % len(servers)]

# Adding server s4 (N=3→4): key "user:42" may land on completely different server
# ~75% of keys remap simultaneously → cache miss storm → DB overload
```

## ✅ After — Consistent Hash Ring: only ~1/N keys move

---
## Full Implementation

In [ ]:
# ❌ BEFORE: hash(key) % N  — everything remaps when N changes
import hashlib

In [ ]:
def naive_assign(key: str, servers: list) -> str:
    return servers[int(hashlib.md5(key.encode()).hexdigest(), 16) % len(servers)]

servers = ["s1", "s2", "s3"]
keys    = [f"user:{i}" for i in range(1000)]

before = {k: naive_assign(k, servers) for k in keys}
after  = {k: naive_assign(k, servers + ["s4"]) for k in keys}  # add ONE server

remapped = sum(1 for k in keys if before[k] != after[k])
print(f"Naive hash%N: adding 1 of 3 nodes remapped {remapped}/{len(keys)} keys "
      f"({remapped/len(keys):.0%})  ← nearly everything!")
# Expected: ~75% remap (3/4 of keys now hash to a different slot)

# ✅ AFTER: Consistent hashing — only ~1/N keys move
import bisect

In [ ]:
def _hash(key: str) -> int:
    return int(hashlib.md5(key.encode()).hexdigest(), 16)

In [ ]:
class Ring:
    def __init__(self, vnodes=150):
        self.vnodes = vnodes
        self._ring: dict[int, str] = {}
        self._sorted: list[int]    = []

    def add(self, server):
        for i in range(self.vnodes):
            pt = _hash(f"{server}#{i}")
            self._ring[pt] = server; bisect.insort(self._sorted, pt)

    def remove(self, server):
        for i in range(self.vnodes):
            pt = _hash(f"{server}#{i}")
            del self._ring[pt]; self._sorted.remove(pt)

    def get(self, key):
        h   = _hash(key)
        idx = bisect.bisect(self._sorted, h) % len(self._sorted)
        return self._ring[self._sorted[idx]]

servers = ["s1", "s2", "s3"]
keys    = [f"user:{i}" for i in range(1000)]

ring = Ring(vnodes=150)
for s in servers: ring.add(s)
before = {k: ring.get(k) for k in keys}

ring.add("s4")   # add one server
after  = {k: ring.get(k) for k in keys}

remapped = sum(1 for k in keys if before[k] != after[k])
print(f"Consistent hashing: adding 1 of 3 nodes remapped {remapped}/{len(keys)} keys "
      f"({remapped/len(keys):.0%})  ← only ~25% (1/N) ✓")

# Verify only s4 received the newly mapped keys
movers = {before[k] for k in keys if before[k] != after[k]}
print(f"  Keys moved FROM: {movers}  (should be mix of old owners handing off to s4)")

"""
04 — System Design: Consistent Hashing (the ring behind sharding & caches)
==========================================================================

Runnable companion to PDF Book V "How do you shard data across N servers?".

The naive way to map a key to one of N servers is `hash(key) % N`. It works —
until N changes. Add or remove one server and (N → N±1) remaps ALMOST EVERY
key, so every cache misses and every shard reshuffles at once: a stampede.

CONSISTENT HASHING fixes this. Servers and keys are placed on the same circular
hash space [0, 2^32). A key belongs to the first server clockwise from it. Add
or remove a server and only the keys in ONE arc move — about 1/N of them.

VIRTUAL NODES (replicas per server) smooth out the otherwise-lumpy distribution.

This file builds the ring and PROVES that removing a node remaps only ~1/N of
keys, versus ~all of them for modulo hashing.
"""

from __future__ import annotations


In [ ]:
def _hash(key: str) -> int:
    return int(hashlib.md5(key.encode()).hexdigest(), 16)

In [ ]:
class ConsistentHashRing:
    def __init__(self, vnodes: int = 100):
        self._vnodes = vnodes
        self._ring: dict[int, str] = {}     # point on ring -> server
        self._sorted: list[int] = []        # sorted ring points for bisect

    def add(self, server: str) -> None:
        for i in range(self._vnodes):
            point = _hash(f"{server}#{i}")  # virtual node
            self._ring[point] = server
            bisect.insort(self._sorted, point)

    def remove(self, server: str) -> None:
        for i in range(self._vnodes):
            point = _hash(f"{server}#{i}")
            del self._ring[point]
            self._sorted.remove(point)

    def get(self, key: str) -> str:
        if not self._sorted:
            raise KeyError("ring is empty")
        h = _hash(key)
        idx = bisect.bisect(self._sorted, h) % len(self._sorted)  # first node clockwise
        return self._ring[self._sorted[idx]]

In [ ]:
def _modulo_assign(key: str, servers: list[str]) -> str:
    return servers[_hash(key) % len(servers)]

In [ ]:
def demo() -> None:
    servers = ["s1", "s2", "s3", "s4", "s5"]
    keys = [f"user:{i}" for i in range(10_000)]

    # --- Consistent hashing: removing one node should move only ~1/N keys. ---
    ring = ConsistentHashRing(vnodes=150)
    for s in servers:
        ring.add(s)
    before = {k: ring.get(k) for k in keys}
    ring.remove("s3")
    after = {k: ring.get(k) for k in keys}
    moved = sum(1 for k in keys if before[k] != after[k])
    frac = moved / len(keys)
    # Only keys that lived on s3 (≈1/5) should move; allow generous slack.
    assert 0.10 < frac < 0.30, f"expected ~1/5 moved, got {frac:.2f}"
    # Every moved key previously belonged to the removed server.
    assert all(before[k] == "s3" for k in keys if before[k] != after[k])
    print(f"   consistent hashing: removing 1 of 5 nodes moved {frac:.1%} of keys (only s3's)")

    # --- Modulo hashing: removing one node remaps almost everything. ---
    mod_before = {k: _modulo_assign(k, servers) for k in keys}
    mod_after = {k: _modulo_assign(k, servers[:-1]) for k in keys}   # drop one server
    mod_moved = sum(1 for k in keys if mod_before[k] != mod_after[k]) / len(keys)
    assert mod_moved > 0.5, "modulo should remap the majority of keys"
    print(f"   modulo hashing:     removing 1 of 5 nodes moved {mod_moved:.1%} of keys (a stampede)")

    # --- Virtual nodes spread load reasonably evenly. ---
    from collections import Counter
    dist = Counter(after.values())
    spread = max(dist.values()) / min(dist.values())
    assert spread < 1.6, f"load should be roughly balanced, spread={spread:.2f}"
    print(f"   virtual nodes: load spread across 4 servers within {spread:.2f}x")

In [ ]:
def main() -> None:
    print("=" * 70)
    print("SYSTEM DESIGN — consistent_hashing.py")
    print("=" * 70)
    print("A hash ring so scaling a cluster moves ~1/N keys, not all of them:")
    demo()
    print("-" * 70)
    print("Lesson: hash(key)%N reshuffles everything when N changes; a consistent-hash ring moves only ~1/N.")
    print("All consistent_hashing demos passed ✔")


if __name__ == "__main__":
    # Keep Unicode output safe even when stdout is redirected/piped (Windows cp1252 fallback).
    import sys
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()